In [2]:
# Download the dataset into the current directory
!wget -nc https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Load the dataset
df = pd.read_csv('course_lead_scoring_2026.csv')

# --- Data Preparation ---
categorical_cols = list(df.select_dtypes(include=['object', 'str']).columns)
numerical_cols = list(df.select_dtypes(exclude=['object', 'str']).columns)

if 'converted' in numerical_cols:
    numerical_cols.remove('converted')

# Replace missing values
df[categorical_cols] = df[categorical_cols].fillna('NA')
df[numerical_cols] = df[numerical_cols].fillna(0.0)

# Split the data with random_state=1 (as required for HW4)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

# Reset indices
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_full_train = df_full_train.reset_index(drop=True)

# Extract targets
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

# Remove targets from dataframes
del df_train['converted']
del df_val['converted']
del df_test['converted']

# ==========================================
# Question 1: ROC AUC feature importance
# ==========================================
q1_features = ['lead_score', 'number_of_courses_viewed', 'interaction_count', 'annual_income']
auc_scores = {}

for f in q1_features:
    auc = roc_auc_score(y_train, df_train[f])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[f])
    auc_scores[f] = auc

best_auc_feature = max(auc_scores, key=auc_scores.get)
print(f"Question 1 Answer: {best_auc_feature}")

# ==========================================
# Question 2: Training the model
# ==========================================
dv = DictVectorizer(sparse=False)
train_dict = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val.to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

# Predict probabilities for validation dataset
y_pred_val = model.predict_proba(X_val)[:, 1]
auc_val = roc_auc_score(y_val, y_pred_val)

print(f"Question 2 Answer: {round(auc_val, 3)}")

# ==========================================
# Question 3: Precision and Recall
# ==========================================
thresholds = np.arange(0.0, 1.01, 0.01)
min_diff = float('inf')
intersect_t = None

for t in thresholds:
    predict_positive = (y_pred_val >= t)
    
    tp = (predict_positive & (y_val == 1)).sum()
    fp = (predict_positive & (y_val == 0)).sum()
    fn = (~predict_positive & (y_val == 1)).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    if precision == 0 and recall == 0:
        continue
        
    diff = abs(precision - recall)
    if diff < min_diff:
        min_diff = diff
        intersect_t = t

print(f"Question 3 Answer: {round(intersect_t, 2)}")

# ==========================================
# Question 4: F1 score
# ==========================================
max_f1 = -1
best_f1_t = None

for t in thresholds:
    predict_positive = (y_pred_val >= t)
    
    tp = (predict_positive & (y_val == 1)).sum()
    fp = (predict_positive & (y_val == 0)).sum()
    fn = (~predict_positive & (y_val == 1)).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    if precision + recall > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0.0
        
    if f1 > max_f1:
        max_f1 = f1
        best_f1_t = t

print(f"Question 4 Answer: {round(best_f1_t, 2)}")

# ==========================================
# Helper functions for CV (Q5 & Q6)
# ==========================================
def train_cv(df_train, y_train, C=1.0):
    dicts = df_train.to_dict(orient='records')
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
    model.fit(X_train, y_train)
    return dv, model

def predict_cv(df, dv, model):
    dicts = df.to_dict(orient='records')
    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]
    return y_pred

# ==========================================
# Question 5: 5-Fold CV
# ==========================================
kfold = KFold(n_splits=5, shuffle=True, random_state=1)
scores_q5 = []

for train_idx, val_idx in kfold.split(df_full_train):
    df_train_cv = df_full_train.iloc[train_idx].copy()
    df_val_cv = df_full_train.iloc[val_idx].copy()
    
    y_train_cv = df_train_cv['converted'].values
    y_val_cv = df_val_cv['converted'].values
    
    del df_train_cv['converted']
    del df_val_cv['converted']
    
    dv_cv, model_cv = train_cv(df_train_cv, y_train_cv, C=1.0)
    y_pred_cv = predict_cv(df_val_cv, dv_cv, model_cv)
    
    auc = roc_auc_score(y_val_cv, y_pred_cv)
    scores_q5.append(auc)

std_auc_q5 = np.std(scores_q5)
print(f"Question 5 Answer: {round(std_auc_q5, 3)}")

# ==========================================
# Question 6: Hyperparameter Tuning
# ==========================================
c_values = [0.000001, 0.001, 1]
best_c = None
best_mean = -1
best_std = float('inf')

for C in c_values:
    scores_q6 = []
    
    for train_idx, val_idx in kfold.split(df_full_train):
        df_train_cv = df_full_train.iloc[train_idx].copy()
        df_val_cv = df_full_train.iloc[val_idx].copy()
        
        y_train_cv = df_train_cv['converted'].values
        y_val_cv = df_val_cv['converted'].values
        
        del df_train_cv['converted']
        del df_val_cv['converted']
        
        dv_cv, model_cv = train_cv(df_train_cv, y_train_cv, C=C)
        y_pred_cv = predict_cv(df_val_cv, dv_cv, model_cv)
        
        auc = roc_auc_score(y_val_cv, y_pred_cv)
        scores_q6.append(auc)
        
    mean_auc = np.mean(scores_q6)
    std_auc = np.std(scores_q6)
    
    mean_rounded = round(mean_auc, 3)
    std_rounded = round(std_auc, 3)
    
    # Tie-breaking logic as described in the instructions
    if mean_rounded > best_mean:
        best_mean = mean_rounded
        best_std = std_rounded
        best_c = C
    elif mean_rounded == best_mean:
        if std_rounded < best_std:
            best_std = std_rounded
            best_c = C
        elif std_rounded == best_std:
            if best_c is None or C < best_c:
                best_c = C

print(f"Question 6 Answer: {best_c}")

--2026-10-19 20:08:37--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.05s   

2026-10-19 20:08:39 (5.28 MB/s) - ‘course_lead_scoring_2026.csv’ saved [278746/278746]

Question 1 Answer: lead_score
Question 2 Answer: 0.732
Question 3 Answer: 0.63
Question 4 Answer: 0.41
Question 5 Answer: 0.007
Question 6 Answer: 0.001
